## Imports

In [117]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
import joblib

import requests

from dotenv import load_dotenv
import os

# Carga de datos desde la API

In [118]:
load_dotenv()
URL_BACKEND_PLAYERS = os.getenv('URL_BACKEND_PLAYERS')

In [119]:
def get_players():
    try:
        response = requests.get(URL_BACKEND_PLAYERS)
        if response.status_code == 200:
            return response.json()
        else:
            print(f'Error al obtener los jugadores: {response.status_code} - {response.text}')
            return None
    except Exception as e:
        print(f'Error con la conexión a la API de jugadores: {e}')
        return None

In [120]:
df = pd.DataFrame(get_players())
print("Datos cargados exitosamente")

Datos cargados exitosamente


In [121]:
df = df.drop(columns=['name'])

# Selección de variables

In [122]:
X = df[['position', 'club_name', 'age', 'last_season', 'league_id', 'country_of_birth', 'height_in_cm']]
y = df['market_value_in_eur']

In [123]:
categorical_features = [
    'position',
    'club_name',
    'league_id',
    'country_of_birth'
]

numerical_features = [
    'age',
    'last_season',
    'height_in_cm'
]

# Preprocesamiento

In [124]:
# ColumnsTransformers permite aplicar transformaiones especificas a columna del dataset
preprocessor = ColumnTransformer(
    transformers=[
        (   
            # Nombre de la rama
            'cat',
            OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-1),
            categorical_features
        ),
        (   
            # Nombre de la rama
            'num',
            'passthrough', # La deja pasar tal cuál, sin realizar ningún cambio
            numerical_features
        )
    ]
)

# División de los datos en conjunto de Entreno y Test

In [125]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Pipeline

In [126]:
# Secuencia de pasos del proceso
pipeline = Pipeline( 
    steps=[ 
        ("preprocessor", preprocessor), 
        (
            # reg:squarederror: meta de de minimizar el error cuadrático medio, n_jobs=-1 es para que el ordenador use todos los núcleos disponibles
            "model", XGBRegressor( objective="reg:squarederror", random_state=42, n_jobs=-1, ),
        ), 
    ]
)

# Grid Search

In [127]:
# Configuración de Hiperparámetros
param_grid = {
    "model__n_estimators": [100, 300, 500], # Nº de árboles de decisión que va a contruir el modelo
    "model__max_depth": [7, 10, 15], # Profundidad de los árboles de decisión
    "model__learning_rate": [0.001, 0.01, 0.05, 0.1], # Tasa de aprendizaje, qué tan rápido aprende el modelo
    # Porcentaje de filas y columnas se seleccional al azar para entrenar cada árbol
    "model__subsample": [0.5, 0.8],
    "model__colsample_bytree": [0.5,0.8, 1.0],
}

# Búsqueda para encontrar la mejor combinación
grid_search = GridSearchCV(
    estimator=pipeline, # El proceso que se va a ejecutar en cada iteración del Pipeline
    param_grid=param_grid,
    cv=5, # Validación Cruzada. DIvisión de los datos de entrenamiento en 5 partes, lo entrena con 4 partes
    scoring="neg_mean_absolute_error", # Métrica para decidir que modelo es mejor: MAE
    n_jobs=-1,
    verbose=2,
    error_score='raise'
)

# Ejecución del entreno de los diferentes modelos.
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__colsample_bytree': [0.5, 0.8, ...], 'model__learning_rate': [0.001, 0.01, ...], 'model__max_depth': [7, 10, ...], 'model__n_estimators': [100, 300, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribut

# Resultado del Grid Search

In [128]:
print("Mejores parámetros encontrados:") 
print(grid_search.best_params_) 
print("\nMejor MAE de validación cruzada:") 
print(-grid_search.best_score_)

Mejores parámetros encontrados:
{'model__colsample_bytree': 1.0, 'model__learning_rate': 0.01, 'model__max_depth': 15, 'model__n_estimators': 300, 'model__subsample': 0.8}

Mejor MAE de validación cruzada:
3267132.95


# Mejor Modelo

In [129]:
best_model = grid_search.best_estimator_

# Predicciones

In [ ]:
predictions = best_model.predict(X_test)

# Métricas

In [131]:
mae = mean_absolute_error(y_test, predictions) 
rmse = mean_squared_error(y_test, predictions) ** 0.5 
r2 = r2_score(y_test, predictions) 

print(f"MAE : {mae:,.2f} €") 
print(f"RMSE: {rmse:,.2f} €") 
print(f"R² : {r2:.4f}")

MAE : 3,221,932.75 €
RMSE: 8,070,228.91 €
R² : 0.5365


In [132]:
best_model = grid_search.best_estimator_

joblib.dump(best_model, "../models/xgboost_model.pkl")

['../models/xgboost_model.pkl']